In [45]:
import os
import sys

# đi tới thư mục project (đúng tên repo của bạn)
PROJECT_ROOT = "/content/Informer2020_tsa"

os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

print("Current working directory:", os.getcwd())
print("Project in sys.path:", PROJECT_ROOT in sys.path)


Current working directory: /content/Informer2020_tsa
Project in sys.path: True


In [52]:
!ls ETDataset


ETT-small  img	LICENSE  README_CN.md  README.md


In [46]:
import argparse
import torch
import os
import io
import sys
import re
import pandas as pd
from contextlib import redirect_stdout

from exp.exp_informer import Exp_Informer


In [ ]:
args = argparse.Namespace()

# ===== Model & data =====
args.model = 'informer'
args.data = 'ETTh1'
args.root_path = './ETDataset/ETT-small/'
args.data_path = 'ETTh1.csv'
args.features = 'M'
args.target = 'OT'
args.freq = 'h'
args.checkpoints = './checkpoints/'

# ===== Sequence length =====
args.seq_len = 96
args.label_len = 48
args.pred_len = 24

# ===== Model size =====
args.enc_in = 7
args.dec_in = 7
args.c_out = 7
args.d_model = 512
args.n_heads = 8
args.e_layers = 2
args.d_layers = 1
args.s_layers = '3,2,1'
args.d_ff = 2048
args.factor = 5
args.padding = 0
args.distil = True
args.dropout = 0.05
args.attn = 'prob'
args.embed = 'timeF'
args.activation = 'gelu'
args.output_attention = False
args.mix = True

# ===== Training =====
args.num_workers = 0
args.itr = 1
args.train_epochs = 6
args.batch_size = 32
args.patience = 3
args.learning_rate = 0.0001
args.des = 'colab_run'
args.loss = 'mse'
args.lradj = 'type1'
args.use_amp = False
args.inverse = False

# ===== GPU =====
args.use_gpu = torch.cuda.is_available()
args.gpu = 0
args.use_multi_gpu = False
args.devices = '0'

args.cols = None



In [48]:
data_parser = {
    'ETTh1':{'data':'ETTh1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTh2':{'data':'ETTh2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm1':{'data':'ETTm1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm2':{'data':'ETTm2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'WTH':{'data':'WTH.csv','T':'WetBulbCelsius','M':[12,12,12],'S':[1,1,1],'MS':[12,12,1]},
    'ECL':{'data':'ECL.csv','T':'MT_320','M':[321,321,321],'S':[1,1,1],'MS':[321,321,1]},
    'Solar':{'data':'solar_AL.csv','T':'POWER_136','M':[137,137,137],'S':[1,1,1],'MS':[137,137,1]},
}

if args.data in data_parser:
    info = data_parser[args.data]
    args.data_path = info['data']
    args.target = info['T']
    args.enc_in, args.dec_in, args.c_out = info[args.features]

args.s_layers = [int(s) for s in args.s_layers.split(',')]
args.detail_freq = args.freq
args.freq = args.freq[-1:]

print("Args in experiment:")
print(args)


Args in experiment:
Namespace(model='informer', data='ETTh1', root_path='./data/ETT/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=24, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8, e_layers=2, d_layers=1, s_layers=[3, 2, 1], d_ff=2048, factor=5, padding=0, distil=True, dropout=0.05, attn='prob', embed='timeF', activation='gelu', output_attention=False, mix=True, num_workers=0, itr=1, train_epochs=6, batch_size=32, patience=3, learning_rate=0.0001, des='colab_run', loss='mse', lradj='type1', use_amp=False, inverse=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', cols=None, detail_freq='h')


In [49]:
pred_len_list = [24, 48, 168, 336, 720]

log_dir = "./logs"
os.makedirs(log_dir, exist_ok=True)

txt_log = os.path.join(log_dir, "pred_len_results.txt")
csv_log = os.path.join(log_dir, "pred_len_results.csv")

with open(txt_log, "w") as f:
    f.write("Informer ETTh1 results\n")
    f.write("pred_len | MSE | MAE\n")
    f.write("=" * 40 + "\n")

results = []
print("Logging to:", txt_log)


Logging to: ./logs/pred_len_results.txt


In [50]:
Exp = Exp_Informer

for pred_len in pred_len_list:
    args.pred_len = pred_len

    setting = f"informer_{args.data}_pl{pred_len}_sl{args.seq_len}_ll{args.label_len}"

    print(f"\n===== Running experiment: pred_len = {pred_len} =====")

    exp = Exp(args)

    # -------- TRAIN --------
    exp.train(setting)

    # -------- TEST (capture stdout) --------
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        exp.test(setting)

    output = buffer.getvalue()

    # -------- PARSE MSE & MAE --------
    mse_match = re.search(r"mse:\s*([0-9\.eE+-]+)", output)
    mae_match = re.search(r"mae:\s*([0-9\.eE+-]+)", output)

    mse = float(mse_match.group(1)) if mse_match else None
    mae = float(mae_match.group(1)) if mae_match else None

    print(f"Result → pred_len={pred_len}, MSE={mse}, MAE={mae}")

    # -------- WRITE TXT LOG --------
    with open(txt_log, "a") as f:
        f.write(f"{pred_len:8d} | {mse:.6f} | {mae:.6f}\n")

    # -------- SAVE CSV DATA --------
    results.append({
        "pred_len": pred_len,
        "MSE": mse,
        "MAE": mae
    })

    torch.cuda.empty_cache()



===== Running experiment: pred_len = 24 =====
Use GPU: cuda:0


FileNotFoundError: [Errno 2] No such file or directory: './data/ETT/ETTh1.csv'

In [ ]:
df = pd.DataFrame(results)
df.to_csv(csv_log, index=False)

df
